# UCF-Crime: download + X3D-S feature extraction

Downloads **the whole UCF-Crime subset (lanes L1 + L2)** into your Google Drive **and extracts X3D-S features from each video as soon as it lands**, so the GPU works while the download is still running.

| Lane | Who | Videos | Size |
|---|---|---|---|
| **L1** | **Colab (default)** | test set + train anomalies (600) | **17.6 GB** |
| **L2** | **Colab (default)** | 400 train normals (seeded random sample) | **26.3 GB** |
| L2X | Colab, optional | the other 400 train normals | 47.1 GB |

**Runtime:** T4 GPU is fastest, but a CPU runtime works too: everything is saved to Drive as it goes, so you can start on CPU and later switch to T4 and *Run all* to resume.

**Output on Drive**

- `MyDrive/vad/ucf_videos/…`: the raw videos
- `MyDrive/vad/features/x3d_s/<video_id>.npy`: `[n_clips, 2048]` float16, plus a `.json` with the metadata

**Resumable:** if Colab disconnects, run all cells again. Finished downloads and finished features are skipped.

The scripts are identical to `scripts/ucf_download.py` and `scripts/extract_x3d.py` in the repo. The manifest is deterministic (seed 415). Only one downloader runs per video folder, so re-running the cells never starts a duplicate.

In [ ]:
# ---- settings: edit before running ----
HF_TOKEN = ""             # optional Hugging Face token; leave empty for anonymous. Clear it before sharing this notebook.
LANES = "L1,L2"           # L1 = test set + train anomalies (600), L2 = 400 train normals; add ",L2X" for the other 400 normals
DOWNLOAD_WORKERS = 12     # parallel downloads
DELETE_AFTER = False      # True = delete each raw video from Drive once its features are saved
DRIVE_ROOT = "/content/drive/MyDrive/vad"

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = HF_TOKEN.strip()
!pip -q install remotezip fvcore
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU: downloading + extracting on CPU (slow but saved as it goes).")
    print("Later: Runtime -> Change runtime type -> T4 GPU, then Run all; it resumes where this stopped.")

### Write the two scripts to `/content/vad_scripts`

In [ ]:
import os
os.makedirs('/content/vad_scripts', exist_ok=True)
UCF_DOWNLOAD_PY = '#!/usr/bin/env python3\n"""Download the UCF-Crime subset used by this project, one video at a time.\n\nVideos are pulled straight out of the Hugging Face mirror\'s zips with HTTP range\nrequests (remotezip), so no full zip is ever downloaded.\n\nLanes (see "Execution Plan.pdf"):\n  L1  Mac    test anomaly (76) + test normal (150) + train anomaly (374) = 600 videos, ~17.6 GB\n  L2  Colab  400 train normals, seeded random sample                       ~37 GB\n  L2X Colab  the other 400 train normals (stretch goal)                     ~37 GB\n\nThe manifest is deterministic (fixed seed), so the Mac and Colab agree on which\nvideos belong to which lane without talking to each other.\n\nRe-running is safe: finished files (correct size) are skipped and partial files\nare restarted. Only one downloader runs per output folder (.downloader.lock);\na second one exits immediately with code 3. An optional Hugging Face token can be set in the HF_TOKEN\nvariable below (or the HF_TOKEN environment variable).\n\nMac usage (download only):\n  caffeinate -i venv/bin/python scripts/ucf_download.py --lanes L1 --workers 8\nDownload + process together: use scripts/extract_x3d.py --with-download (it starts this script).\n"""\nimport argparse\nimport csv\nimport os\nimport random\nimport shutil\nimport socket\nimport sys\nimport threading\nimport time\nimport urllib.request\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\n\nfrom remotezip import RemoteZip\n\n# Optional Hugging Face token (helps only if you hit HTTP 429 rate limits).\n# Paste it between the quotes before running; leave empty for anonymous access.\n# Don\'t share or commit this file with the token filled in.\nHF_TOKEN = ""\n\nBASE = "https://huggingface.co/datasets/jinmang2/ucf_crime/resolve/main/"\nZIPS = [\n    "Anomaly-Videos-Part-1.zip",\n    "Anomaly-Videos-Part-2.zip",\n    "Anomaly-Videos-Part-3.zip",\n    "Anomaly-Videos-Part-4.zip",\n    "Testing_Normal_Videos.zip",\n    "Training-Normal-Videos-Part-1.zip",\n    "Training-Normal-Videos-Part-2.zip",\n]\nSPLIT_FILES = {\n    "Anomaly_Train.txt": "UCF_Crimes-Train-Test-Split/Anomaly_Detection_splits/Anomaly_Train.txt",\n    "Anomaly_Test.txt": "UCF_Crimes-Train-Test-Split/Anomaly_Detection_splits/Anomaly_Test.txt",\n    "Temporal_Anomaly_Annotation_for_Testing_Videos.txt":\n        "UCF_Crimes-Train-Test-Split/Temporal_Anomaly_Annotation_for_Testing_Videos.txt",\n}\nCLASSES = {"Arson", "Burglary", "Explosion", "Fighting", "Robbery", "Shooting"}\nSEED = 415\nN_TRAIN_NORMAL = 400\nMANIFEST_FIELDS = ["video_id", "split", "label", "cls", "lane", "zip", "member", "bytes", "rel_path"]\n\n\ndef hf_headers():\n    token = (HF_TOKEN or os.environ.get("HF_TOKEN", "")).strip()\n    h = {"User-Agent": "ucf-subset-downloader (research)"}\n    if token:\n        h["Authorization"] = f"Bearer {token}"\n    return h\n\n\ndef ensure_splits(split_dir):\n    os.makedirs(split_dir, exist_ok=True)\n    for name, remote in SPLIT_FILES.items():\n        path = os.path.join(split_dir, name)\n        if os.path.exists(path) and os.path.getsize(path) > 0:\n            continue\n        req = urllib.request.Request(BASE + remote, headers=hf_headers())\n        with urllib.request.urlopen(req, timeout=60) as r, open(path, "wb") as f:\n            f.write(r.read())\n        print(f"fetched {name}")\n\n\ndef zip_index():\n    """basename -> (zip, member, size) for every mp4 in the relevant zips."""\n    idx = {}\n    for z in ZIPS:\n        with RemoteZip(BASE + z, headers=hf_headers()) as rz:\n            for info in rz.infolist():\n                if not info.filename.endswith(".mp4"):\n                    continue\n                b = os.path.basename(info.filename)\n                if b in idx:\n                    raise RuntimeError(f"duplicate basename {b} in {z} and {idx[b][0]}")\n                idx[b] = (z, info.filename, info.file_size)\n    return idx\n\n\ndef build_manifest(split_dir, manifest_path):\n    ensure_splits(split_dir)\n    idx = zip_index()\n    rows, train_normals = [], []\n    for split, fname in (("train", "Anomaly_Train.txt"), ("test", "Anomaly_Test.txt")):\n        for line in open(os.path.join(split_dir, fname)):\n            p = line.strip()\n            if not p:\n                continue\n            folder, b = p.split("/")[0], os.path.basename(p)\n            if folder in CLASSES:\n                cls, label = folder, 1\n            elif "Normal" in folder:\n                cls, label = "Normal", 0\n            else:\n                continue\n            z, member, size = idx[b]\n            row = {\n                "video_id": b[:-4], "split": split, "label": label, "cls": cls, "lane": "",\n                "zip": z, "member": member, "bytes": size,\n                "rel_path": f"{split.capitalize()}/{cls}/{b}",\n            }\n            if split == "train" and label == 0:\n                train_normals.append(row)\n            else:\n                row["lane"] = "L1"\n                rows.append(row)\n    train_normals.sort(key=lambda r: r["video_id"])\n    chosen = set(r["video_id"] for r in random.Random(SEED).sample(train_normals, N_TRAIN_NORMAL))\n    for r in train_normals:\n        r["lane"] = "L2" if r["video_id"] in chosen else "L2X"\n        rows.append(r)\n    os.makedirs(os.path.dirname(manifest_path) or ".", exist_ok=True)\n    with open(manifest_path, "w", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=MANIFEST_FIELDS)\n        w.writeheader()\n        w.writerows(rows)\n    return rows\n\n\ndef load_or_build_manifest(split_dir, manifest_path):\n    if os.path.exists(manifest_path):\n        with open(manifest_path) as f:\n            rows = list(csv.DictReader(f))\n        for r in rows:\n            r["bytes"] = int(r["bytes"])\n            r["label"] = int(r["label"])\n        return rows\n    print("building manifest (lists the 7 remote zips, ~30 s)...")\n    return build_manifest(split_dir, manifest_path)\n\n\n_local = threading.local()\n\n\ndef _zip_handle(z):\n    cache = getattr(_local, "zips", None)\n    if cache is None:\n        cache = _local.zips = {}\n    if z not in cache:\n        cache[z] = RemoteZip(BASE + z, headers=hf_headers())\n    return cache[z]\n\n\ndef _drop_handle(z):\n    cache = getattr(_local, "zips", {})\n    rz = cache.pop(z, None)\n    if rz is not None:\n        try:\n            rz.close()\n        except Exception:\n            pass\n\n\ndef download_one(row, out_root, retries=6):\n    dest = os.path.join(out_root, row["rel_path"])\n    if os.path.exists(dest) and os.path.getsize(dest) == row["bytes"]:\n        return "skip", 0\n    os.makedirs(os.path.dirname(dest), exist_ok=True)\n    tmp = dest + ".part"\n    for attempt in range(retries):\n        try:\n            rz = _zip_handle(row["zip"])\n            with rz.open(row["member"]) as src, open(tmp, "wb") as dst:  # zipfile checks the CRC at EOF\n                shutil.copyfileobj(src, dst, 4 << 20)\n            got = os.path.getsize(tmp)\n            if got != row["bytes"]:\n                raise IOError(f"size mismatch {got} != {row[\'bytes\']}")\n            os.replace(tmp, dest)\n            return "ok", row["bytes"]\n        except Exception as e:  # network hiccup, 429, CRC error...\n            _drop_handle(row["zip"])\n            wait = min(120, 5 * 2 ** attempt)\n            print(f"  retry {attempt + 1}/{retries} {row[\'video_id\']}: {type(e).__name__}: {e} (sleep {wait}s)",\n                  flush=True)\n            time.sleep(wait)\n    if os.path.exists(tmp):\n        os.remove(tmp)\n    return "fail", 0\n\n\ndef run(rows, out_root, workers, log_path=None, check_disk=True):\n    todo = [r for r in rows\n            if not (os.path.exists(os.path.join(out_root, r["rel_path"]))\n                    and os.path.getsize(os.path.join(out_root, r["rel_path"])) == r["bytes"])]\n    total_b = sum(r["bytes"] for r in todo)\n    print(f"{len(rows)} videos in selection, {len(rows) - len(todo)} already done, "\n          f"{len(todo)} to fetch ({total_b / 1e9:.2f} GB) with {workers} workers", flush=True)\n    if not todo:\n        return []\n    free = shutil.disk_usage(out_root if os.path.exists(out_root) else ".").free\n    if check_disk and free < total_b + 2e9:\n        sys.exit(f"not enough disk: need {total_b / 1e9:.1f} GB + 2 GB headroom, have {free / 1e9:.1f} GB")\n    t0, done_b, n_done, failed = time.time(), 0, 0, []\n    log = open(log_path, "a") if log_path else None\n    with ThreadPoolExecutor(max_workers=workers) as ex:\n        futs = {ex.submit(download_one, r, out_root): r for r in todo}\n        for fut in as_completed(futs):\n            r = futs[fut]\n            status, nb = fut.result()\n            n_done += 1\n            done_b += nb\n            if status == "fail":\n                failed.append(r["video_id"])\n            el = time.time() - t0\n            rate = done_b / el if el else 0\n            eta = (total_b - done_b) / rate / 60 if rate else float("nan")\n            msg = (f"[{n_done}/{len(todo)}] {status:4s} {r[\'rel_path\']} {r[\'bytes\'] / 1e6:.0f} MB | "\n                   f"{done_b / 1e9:.2f}/{total_b / 1e9:.2f} GB  {rate / 1e6:.1f} MB/s  ETA {eta:.0f} min")\n            print(msg, flush=True)\n            if log:\n                log.write(msg + "\\n")\n                log.flush()\n    print(f"finished in {(time.time() - t0) / 60:.1f} min; failed: {len(failed)} {failed[:10]}", flush=True)\n    return failed\n\n\nLOCK_NAME = ".downloader.lock"\nLOCK_FRESH = 120  # seconds; the holder touches the lock every 30 s\nEXIT_ALREADY_RUNNING = 3\n\n\ndef _pid_alive(pid):\n    try:\n        os.kill(pid, 0)\n        return True\n    except ProcessLookupError:\n        return False\n    except PermissionError:\n        return True\n\n\ndef lock_is_fresh(out_root):\n    p = os.path.join(out_root, LOCK_NAME)\n    try:\n        return time.time() - os.path.getmtime(p) < LOCK_FRESH\n    except OSError:\n        return False\n\n\ndef acquire_lock(out_root):\n    """One downloader per output folder. Returns False if another live one holds it."""\n    p = os.path.join(out_root, LOCK_NAME)\n    host = socket.gethostname()\n    if lock_is_fresh(out_root):\n        try:\n            h, pid = open(p).read().split()[:2]\n            if h != host or _pid_alive(int(pid)):\n                return False\n        except Exception:\n            pass\n    with open(p, "w") as f:\n        f.write(f"{host} {os.getpid()}\\n")\n\n    def beat():\n        while True:\n            time.sleep(30)\n            try:\n                os.utime(p)\n            except OSError:\n                pass\n    threading.Thread(target=beat, daemon=True).start()\n    return True\n\n\ndef release_lock(out_root):\n    p = os.path.join(out_root, LOCK_NAME)\n    try:\n        if open(p).read().split()[1] == str(os.getpid()):\n            os.remove(p)\n    except Exception:\n        pass\n\n\ndef main():\n    here = os.path.dirname(os.path.abspath(__file__))\n    proj = os.path.dirname(here)\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--lanes", default="L1", help="comma list of L1,L2,L2X")\n    ap.add_argument("--out", default=os.path.join(proj, "ucf_videos.nosync"), help="video output root")\n    ap.add_argument("--splits", default=os.path.join(proj, "data", "splits"))\n    ap.add_argument("--manifest", default=os.path.join(proj, "data", "manifests", "ucf_subset.csv"))\n    ap.add_argument("--workers", type=int, default=4)\n    ap.add_argument("--limit", type=int, default=0, help="only the first N videos (testing)")\n    ap.add_argument("--no-disk-check", action="store_true", help="skip the free-space check (use on a Drive mount)")\n    a = ap.parse_args()\n    rows = load_or_build_manifest(a.splits, a.manifest)\n    lanes = set(a.lanes.split(","))\n    sel = [r for r in rows if r["lane"] in lanes]\n    order = {"test": 0, "train": 1}\n    sel.sort(key=lambda r: (order[r["split"]], -r["label"], r["video_id"]))  # eval set first\n    if a.limit:\n        sel = sel[:a.limit]\n    os.makedirs(a.out, exist_ok=True)\n    if not acquire_lock(a.out):\n        print(f"another downloader is already running on {a.out} (see {LOCK_NAME}); not starting a second one.",\n              flush=True)\n        sys.exit(EXIT_ALREADY_RUNNING)\n    try:\n        failed = run(sel, a.out, a.workers, log_path=os.path.join(a.out, "download_log.txt"),\n                     check_disk=not a.no_disk_check)\n    finally:\n        release_lock(a.out)\n    sys.exit(1 if failed else 0)\n\n\nif __name__ == "__main__" and "ipykernel" not in sys.modules:\n    main()\n'
EXTRACT_X3D_PY = '#!/usr/bin/env python3\n"""Extract X3D-S clip features from UCF-Crime videos *while they download*.\n\nIt watches the download folder and processes each video as soon as the\ndownloader finishes it (the downloader writes `<name>.part` and renames it when\ncomplete, so a video with its final name and the manifest size is safe to read).\nWith --with-download it also starts scripts/ucf_download.py for you, so one\ncommand does both and the GPU/MPS never waits for the whole download.\n\nPer video it writes:\n  <out>/<video_id>.npy   float16 [n_clips, 2048]  X3D-S penultimate features\n  <out>/<video_id>.json  fps, n_frames, n_clips, clip/stride settings, timing\n\nClip definition (fixed, identical on Mac and Colab): every 6th frame is kept,\n13 kept frames form one clip, so clip i covers raw frames [78*i, 78*i + 78)\n(~2.6 s at 30 fps), non-overlapping; the last clip is padded with its final frame.\nFrames are resized (short side 182), centre-cropped to 182x182 and normalised\nwith mean 0.45 / std 0.225 (PyTorchVideo X3D defaults).\n\nSafe to stop and re-run: finished videos are skipped. A video in progress is\ncheckpointed every --ckpt-secs to <out>/_partial/<video_id>.npy, so a rerun (even\non a different machine or runtime, e.g. Colab CPU -> T4) continues from the last\nsaved clip instead of restarting the video. Claim files let several workers\nshare one feature folder without doing the same video twice.\n\nMac (download + process, one command):\n  caffeinate -i venv/bin/python scripts/extract_x3d.py --lanes L1 --with-download\nProcess only (videos already downloaded, or downloader running elsewhere):\n  venv/bin/python scripts/extract_x3d.py --lanes L1 --wait\n\nFirst run needs:  pip install fvcore   (X3D is loaded via torch.hub from PyTorchVideo)\n"""\nimport argparse\nimport csv\nimport json\nimport os\nimport socket\nimport subprocess\nimport sys\nimport time\n\nimport cv2\nimport numpy as np\nimport torch\n\nSTRIDE = 6          # keep every 6th raw frame\nCLIP_LEN = 13       # kept frames per clip  -> 78 raw frames per clip\nSIZE = 182          # X3D-S crop size\nMEAN, STD = 0.45, 0.225\nCLAIM_TTL = 20 * 60  # seconds before another worker may take over a claimed video\n\n\n# ---------------------------------------------------------------- model\ndef pick_device(name):\n    if name != "auto":\n        return torch.device(name)\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef load_x3d(device):\n    model = torch.hub.load("facebookresearch/pytorchvideo", "x3d_s", pretrained=True, trust_repo=True)\n    head = model.blocks[-1]\n    head.proj = torch.nn.Identity()          # drop the 400-way Kinetics classifier\n    if getattr(head, "activation", None) is not None:\n        head.activation = torch.nn.Identity()\n    model = model.eval().to(device)\n    dtype = torch.float16 if device.type in ("cuda", "mps") else torch.float32\n    model = model.to(dtype)\n    with torch.inference_mode():\n        d = model(torch.zeros(1, 3, CLIP_LEN, SIZE, SIZE, device=device, dtype=dtype)).shape[-1]\n    if d != 2048:\n        raise RuntimeError(f"expected 2048-d features, got {d}")\n    return model, dtype\n\n\n# ---------------------------------------------------------------- video\ndef _prep(frame):\n    h, w = frame.shape[:2]\n    s = SIZE / min(h, w)\n    frame = cv2.resize(frame, (max(SIZE, round(w * s)), max(SIZE, round(h * s))), interpolation=cv2.INTER_AREA)\n    h, w = frame.shape[:2]\n    y, x = (h - SIZE) // 2, (w - SIZE) // 2\n    return cv2.cvtColor(frame[y:y + SIZE, x:x + SIZE], cv2.COLOR_BGR2RGB)\n\n\ndef iter_clip_batches(path, batch, skip_clips=0):\n    """Yields uint8 arrays [b, 13, 182, 182, 3]; returns (n_raw_frames, fps) via StopIteration value.\n\n    skip_clips: the first N clips are already embedded (resume); their frames are\n    only grabbed (cheap), not decoded to images or yielded.\n    """\n    cap = cv2.VideoCapture(path)\n    if not cap.isOpened():\n        raise IOError("cannot open video")\n    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0\n    n_raw, clip, clips = 0, [], []\n    skip_frames = skip_clips * CLIP_LEN * STRIDE\n    while True:\n        if not cap.grab():\n            break\n        if n_raw < skip_frames:\n            n_raw += 1\n            continue\n        if n_raw % STRIDE == 0:\n            ok, frame = cap.retrieve()\n            if ok:\n                clip.append(_prep(frame))\n            if len(clip) == CLIP_LEN:\n                clips.append(np.stack(clip))\n                clip = []\n                if len(clips) == batch:\n                    yield np.stack(clips)\n                    clips = []\n        n_raw += 1\n    cap.release()\n    if clip:  # pad the final partial clip with its last frame\n        clip += [clip[-1]] * (CLIP_LEN - len(clip))\n        clips.append(np.stack(clip))\n    if clips:\n        yield np.stack(clips)\n    return n_raw, fps\n\n\ndef load_partial(partial_path):\n    """Features saved by an interrupted run, or None."""\n    if not os.path.exists(partial_path):\n        return None\n    try:\n        arr = np.load(partial_path)\n        if arr.ndim == 2 and arr.shape[1] == 2048 and len(arr) > 0 and np.isfinite(arr).all():\n            return arr.astype(np.float32)\n    except Exception:\n        pass\n    os.remove(partial_path)  # unreadable / wrong shape: start the video over\n    return None\n\n\ndef save_partial(partial_path, feats):\n    tmp = partial_path[:-4] + ".tmp.npy"\n    np.save(tmp, np.concatenate(feats).astype(np.float16))\n    os.replace(tmp, partial_path)\n\n\ndef embed_video(model, dtype, device, path, batch, partial_path=None, ckpt_secs=120):\n    """Returns (feats fp16 [n_clips, 2048], n_raw_frames, fps, resumed_clips)."""\n    prev = load_partial(partial_path) if partial_path else None\n    feats = [prev] if prev is not None else []\n    skip = len(prev) if prev is not None else 0\n    gen = iter_clip_batches(path, batch, skip_clips=skip)\n    last_ckpt = time.time()\n    mean = torch.tensor(MEAN, device=device, dtype=dtype)\n    std = torch.tensor(STD, device=device, dtype=dtype)\n    while True:\n        try:\n            arr = next(gen)\n        except StopIteration as stop:\n            n_raw, fps = stop.value\n            break\n        x = torch.from_numpy(arr).to(device)                       # b,t,h,w,c uint8\n        x = x.permute(0, 4, 1, 2, 3).to(dtype).div_(255).sub_(mean).div_(std)  # b,c,t,h,w\n        with torch.inference_mode():\n            feats.append(model(x).float().cpu().numpy())\n        if partial_path and time.time() - last_ckpt >= ckpt_secs:\n            save_partial(partial_path, feats)\n            last_ckpt = time.time()\n    if not feats:\n        raise IOError("no frames decoded")\n    out = np.concatenate(feats).astype(np.float16)\n    expected = -(-n_raw // (CLIP_LEN * STRIDE))  # ceil\n    if skip and len(out) != expected:\n        raise IOError(f"clip count {len(out)} != expected {expected} (stale partial?)")\n    return out, n_raw, fps, skip\n\n\n# ---------------------------------------------------------------- bookkeeping\ndef load_rows(manifest, lanes):\n    with open(manifest) as f:\n        rows = [r for r in csv.DictReader(f) if r["lane"] in lanes]\n    order = {"test": 0, "train": 1}\n    rows.sort(key=lambda r: (order[r["split"]], -int(r["label"]), r["video_id"]))  # same order as the downloader\n    return rows\n\n\ndef video_ready(path, size):\n    return os.path.exists(path) and os.path.getsize(path) == size\n\n\ndef try_claim(claim_dir, vid, worker):\n    p = os.path.join(claim_dir, vid + ".claim")\n    if os.path.exists(p) and time.time() - os.path.getmtime(p) < CLAIM_TTL:\n        try:\n            if open(p).read().split()[0] != worker:\n                return False\n        except Exception:\n            return False\n    with open(p, "w") as f:\n        f.write(f"{worker} {time.time():.0f}\\n")\n    return True\n\n\ndef save_atomic(out_dir, vid, feats, meta):\n    tmp = os.path.join(out_dir, vid + ".tmp.npy")\n    np.save(tmp, feats)\n    os.replace(tmp, os.path.join(out_dir, vid + ".npy"))\n    with open(os.path.join(out_dir, vid + ".json.tmp"), "w") as f:\n        json.dump(meta, f)\n    os.replace(os.path.join(out_dir, vid + ".json.tmp"), os.path.join(out_dir, vid + ".json"))\n\n\n# ---------------------------------------------------------------- main loop\ndef main():\n    here = os.path.dirname(os.path.abspath(__file__))\n    proj = os.path.dirname(here)\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument("--lanes", default="L1", help="comma list of L1,L2,L2X")\n    ap.add_argument("--videos", default=os.path.join(proj, "ucf_videos.nosync"), help="download root")\n    ap.add_argument("--manifest", default=os.path.join(proj, "data", "manifests", "ucf_subset.csv"))\n    ap.add_argument("--splits", default=os.path.join(proj, "data", "splits"))\n    ap.add_argument("--out", default=os.path.join(proj, "data", "features", "x3d_s"))\n    ap.add_argument("--device", default="auto", help="auto / mps / cuda / cpu")\n    ap.add_argument("--batch", type=int, default=0, help="clips per forward pass (default 16 mps, 32 cuda, 8 cpu)")\n    ap.add_argument("--worker", default=socket.gethostname().split(".")[0])\n    ap.add_argument("--with-download", action="store_true", help="also start ucf_download.py for the same lanes")\n    ap.add_argument("--download-workers", type=int, default=8)\n    ap.add_argument("--no-disk-check", action="store_true", help="passed to the downloader (Drive mounts)")\n    ap.add_argument("--wait", action="store_true", help="keep polling for new videos until the lanes are complete")\n    ap.add_argument("--poll", type=int, default=20, help="seconds between checks while waiting")\n    ap.add_argument("--idle-timeout", type=int, default=45 * 60,\n                    help="with --wait and no --with-download: stop after this many seconds without new videos")\n    ap.add_argument("--delete-after", action="store_true", help="delete each video once its features are saved")\n    ap.add_argument("--limit", type=int, default=0, help="stop after N videos (testing)")\n    ap.add_argument("--ckpt-secs", type=int, default=120,\n                    help="save a video\'s partial features this often, so an interrupted video resumes mid-way")\n    a = ap.parse_args()\n    lanes = set(a.lanes.split(","))\n\n    dl = None\n    if a.with_download:\n        cmd = [sys.executable, "-u", os.path.join(here, "ucf_download.py"), "--lanes", a.lanes, "--out", a.videos,\n               "--splits", a.splits, "--manifest", a.manifest, "--workers", str(a.download_workers)]\n        if a.no_disk_check:\n            cmd.append("--no-disk-check")\n        os.makedirs(a.videos, exist_ok=True)\n        dl_log = open(os.path.join(a.videos, "downloader_stdout.txt"), "a")\n        dl = subprocess.Popen(cmd, stdout=dl_log, stderr=subprocess.STDOUT)\n        print(f"downloader started (pid {dl.pid}), log: {dl_log.name}", flush=True)\n        a.wait = True\n\n    def downloading():\n        """Our downloader child is alive, or another one (e.g. from an earlier,\n        interrupted run) holds a fresh lock on the video folder."""\n        if dl is not None and dl.poll() is None:\n            return True\n        try:\n            return time.time() - os.path.getmtime(os.path.join(a.videos, ".downloader.lock")) < 120\n        except OSError:\n            return False\n\n    # the downloader builds the manifest on first run; wait for it\n    while not os.path.exists(a.manifest):\n        if dl is not None and not downloading():\n            sys.exit("downloader exited before writing the manifest; see downloader_stdout.txt")\n        if dl is None:\n            sys.exit(f"manifest not found: {a.manifest} (run ucf_download.py once, or use --with-download)")\n        time.sleep(3)\n\n    rows = load_rows(a.manifest, lanes)\n    os.makedirs(a.out, exist_ok=True)\n    claim_dir = os.path.join(a.out, "_claims")\n    os.makedirs(claim_dir, exist_ok=True)\n    partial_dir = os.path.join(a.out, "_partial")\n    os.makedirs(partial_dir, exist_ok=True)\n    log = open(os.path.join(a.out, f"_log_{a.worker}.jsonl"), "a")\n\n    device = pick_device(a.device)\n    batch = a.batch or {"cuda": 32, "mps": 16}.get(device.type, 8)\n    print(f"loading X3D-S on {device} (batch {batch})...", flush=True)\n    model, dtype = load_x3d(device)\n\n    n_done_run, t_start, clips_run, model_secs = 0, time.time(), 0, 0.0\n    last_progress = time.time()\n    failures = {}\n    while True:\n        done = pending = 0\n        progressed = False\n        for r in rows:\n            vid = r["video_id"]\n            if os.path.exists(os.path.join(a.out, vid + ".npy")):\n                done += 1\n                continue\n            path = os.path.join(a.videos, r["rel_path"])\n            if not video_ready(path, int(r["bytes"])) or failures.get(vid, 0) >= 2:\n                pending += 1\n                continue\n            if not try_claim(claim_dir, vid, a.worker):\n                pending += 1\n                continue\n            t0 = time.time()\n            partial_path = os.path.join(partial_dir, vid + ".npy")\n            try:\n                feats, n_raw, fps, resumed = embed_video(model, dtype, device, path, batch,\n                                                         partial_path, a.ckpt_secs)\n            except Exception as e:\n                failures[vid] = failures.get(vid, 0) + 1\n                if os.path.exists(partial_path):\n                    os.remove(partial_path)  # retry this video from scratch\n                print(f"  FAIL {vid}: {type(e).__name__}: {e}", flush=True)\n                log.write(json.dumps({"video_id": vid, "status": "error", "error": str(e)}) + "\\n")\n                log.flush()\n                pending += 1\n                continue\n            dt = time.time() - t0\n            meta = {"video_id": vid, "split": r["split"], "label": int(r["label"]), "cls": r["cls"],\n                    "fps": fps, "n_frames": n_raw, "n_clips": int(feats.shape[0]), "stride": STRIDE,\n                    "clip_len": CLIP_LEN, "size": SIZE, "worker": a.worker, "device": device.type,\n                    "seconds": round(dt, 2), "resumed_clips": resumed}\n            save_atomic(a.out, vid, feats, meta)\n            for p in (os.path.join(claim_dir, vid + ".claim"), partial_path):\n                try:\n                    os.remove(p)\n                except OSError:\n                    pass\n            if a.delete_after:\n                os.remove(path)\n            log.write(json.dumps({**meta, "status": "ok"}) + "\\n")\n            log.flush()\n            done += 1\n            n_done_run += 1\n            new_clips = feats.shape[0] - resumed\n            clips_run += new_clips\n            model_secs += dt\n            progressed = True\n            note = f"  (resumed at clip {resumed})" if resumed else ""\n            print(f"[{done}/{len(rows)}] {vid:28s} {feats.shape[0]:5d} clips  {dt:6.1f}s  "\n                  f"{1000 * dt / max(new_clips, 1):5.1f} ms/clip{note}", flush=True)\n            if a.limit and n_done_run >= a.limit:\n                break\n        if a.limit and n_done_run >= a.limit:\n            break\n        if done == len(rows):\n            print("all videos in the selected lanes have features.", flush=True)\n            break\n        if progressed:\n            last_progress = time.time()\n        dl_running = downloading()\n        idle_out = dl is None and not dl_running and time.time() - last_progress > a.idle_timeout\n        if not a.wait or idle_out or (dl is not None and not dl_running and not progressed):\n            print(f"stopping: {done}/{len(rows)} done, {pending} not available "\n                  f"(not downloaded, failed, or claimed by another worker).", flush=True)\n            break\n        if not progressed:\n            print(f"  waiting for downloads... {done}/{len(rows)} done, {pending} pending", flush=True)\n            time.sleep(a.poll)\n\n    el = time.time() - t_start\n    if clips_run:\n        print(f"this run: {n_done_run} videos, {clips_run} clips, {el / 60:.1f} min wall, "\n              f"{1000 * model_secs / clips_run:.1f} ms/clip incl. decode", flush=True)\n    if failures:\n        print("failed videos:", sorted(failures), flush=True)\n    if downloading():\n        print("downloader still running; leaving it to finish.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
open('/content/vad_scripts/ucf_download.py', 'w').write(UCF_DOWNLOAD_PY)
open('/content/vad_scripts/extract_x3d.py', 'w').write(EXTRACT_X3D_PY)
print('scripts written')

### Run: download + extract together

It starts the downloader in the background, then processes videos as they finish, on the GPU if the runtime has one, otherwise on the CPU. The downloader's own progress is in `MyDrive/vad/ucf_videos/download_log.txt`. If this cell is interrupted, the background download keeps going. Re-run the cell to resume extraction.

**Switching runtime (e.g. CPU now, T4 later):** stop this cell, run the **flush** cell at the bottom, then change the runtime and *Run all*. Finished videos, finished features and the partial features of the video in progress (`features/x3d_s/_partial/`, saved every 2 min) are all on Drive, so it carries on from the last saved clip.

In [ ]:
extra = "--delete-after" if DELETE_AFTER else ""
!python /content/vad_scripts/extract_x3d.py --lanes {LANES} --with-download --no-disk-check \
    --download-workers {DOWNLOAD_WORKERS} --worker colab --device auto {extra} \
    --videos {DRIVE_ROOT}/ucf_videos --splits {DRIVE_ROOT}/splits \
    --manifest {DRIVE_ROOT}/manifests/ucf_subset.csv --out {DRIVE_ROOT}/features/x3d_s

In [ ]:
# ---- progress check (safe to run any time) ----
import csv, os, collections
rows = list(csv.DictReader(open(f"{DRIVE_ROOT}/manifests/ucf_subset.csv")))
vid_ok, feat_ok, need = collections.Counter(), collections.Counter(), collections.Counter()
for r in rows:
    need[r["lane"]] += 1
    p = os.path.join(DRIVE_ROOT, "ucf_videos", r["rel_path"])
    if os.path.exists(p) and os.path.getsize(p) == int(r["bytes"]): vid_ok[r["lane"]] += 1
    if os.path.exists(f"{DRIVE_ROOT}/features/x3d_s/{r['video_id']}.npy"): feat_ok[r["lane"]] += 1
for lane in sorted(need):
    print(f"{lane}: videos {vid_ok[lane]}/{need[lane]}   features {feat_ok[lane]}/{need[lane]}")
!tail -3 {DRIVE_ROOT}/ucf_videos/download_log.txt

In [ ]:
# ---- when everything is finished: flush all writes to Drive before the runtime is recycled ----
drive.flush_and_unmount()
print("flushed")